In [ ]:
!pip install gensim
from google.colab import drive
import pandas as pd
import numpy as np
import os
from datetime import datetime
import re
import gc
import spacy

from html import unescape
from gensim.corpora import Dictionary
from gensim.models.phrases import Phrases, Phraser
from gensim import matutils
from sklearn.decomposition import LatentDirichletAllocation
drive.mount('/content/drive')

CHECKPOINT_PATH = "/content/drive/MyDrive/Thesis Code/llm_extracted_checkpoint_2013_2019V2.parquet"

#dates should be the same as used in main
start_date = datetime(2013, 1, 1)
end_date = datetime(2019, 12, 31)

In [ ]:
#get data
df = pd.read_parquet(CHECKPOINT_PATH).reset_index(drop=True)

In [ ]:
# config
RANDOM_STATE = 42
TEST_FRACTION = 0.20

N_TOPIC_CANDIDATES = [5, 8, 10, 12, 15, 20, 25, 30]
LDA_SELECTION_N_DOCS = 75_000
LDA_EVAL_N_DOCS = 20_000
LDA_TOP_WORDS = 10
LDA_SELECTION_MAX_ITER = 5
LDA_BATCH_SIZE = 2048

PHRASE_MIN_COUNT = 10
PHRASE_THRESHOLD = 10

VOCAB_NO_BELOW = 5
VOCAB_NO_ABOVE = 0.40
LDA_MAX_FEATURES = None

SPACY_BATCH_SIZE = 1000
SPACY_N_PROCESS = -1

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

custom_stopwords = {
    "thing", "stuff", "item", "something", "anything", "product", "products",
    "use", "used", "using", "get", "got", "make", "makes", "made",
    "work", "works", "tried", "try", "really", "just", "also", "much",
    "one", "time", "way", "little", "don", "doesn", "didn", "isn",
    "wasn", "weren", "wouldn", "couldn", "shouldn", "hasn", "haven",
    "aren", "purchase", "purchased", "purchasing", "friend", "family",
    "especially", "want", "wanted", "think", "thought", "know", "knew",
    "like",
}

stop_words = set(nlp.Defaults.stop_words) | custom_stopwords

# some functions for preprocessing
def clean_html(text):
    text = unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def basic_clean(text):
    text = clean_html(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

from tqdm.auto import tqdm
def process_docs(texts, batch_size=2000, n_process=-2):
    processed = []

    docs = nlp.pipe(
        texts,
        batch_size=batch_size,
        n_process=n_process
    )

    for doc in tqdm(docs, total=len(texts), desc="spaCy preprocessing"):
        processed.append([
            token.lemma_.lower()
            for token in doc
            if token.is_alpha
            and len(token.lemma_) > 2
            and token.lemma_.lower() not in stop_words
        ])

    return processed

def corpus_to_csr(corpus, n_terms):
    return matutils.corpus2csc(
        corpus,
        num_terms=n_terms,
        dtype=np.float32
    ).T.tocsr()

def remove_empty_documents(tokens, corpus):
    valid = [i for i, doc in enumerate(corpus) if len(doc) > 0]
    return [tokens[i] for i in valid], [corpus[i] for i in valid]

In [ ]:
# prepare mask used so only fit/evaluate the results with training data

all_months = np.array(sorted(df["year_month"].dropna().unique()))
split_idx = int(len(all_months) * (1 - TEST_FRACTION))
split_idx = min(max(split_idx, 1), len(all_months) - 1)

train_cutoff = all_months[split_idx]
train_reviews_mask = df["year_month"] < train_cutoff
test_reviews_mask = df["year_month"] >= train_cutoff

print(f"Chronological cutoff: {train_cutoff}")
print(f"Training review rows: {train_reviews_mask.sum():,}")
print(f"Holdout review rows:  {test_reviews_mask.sum():,}")
print(f"Training months:      {df.loc[train_reviews_mask, 'year_month'].nunique():,}")
print(f"Holdout months:       {df.loc[test_reviews_mask, 'year_month'].nunique():,}")


In [ ]:
# RANDOM 75K SELECTION + 20K EVALUATION SAMPLE
train_text = (
    df.loc[train_reviews_mask, "review_text"]
      .fillna("")
      .astype(str)
      .reset_index(drop=True)
)

n_available = len(train_text)
required_docs = LDA_SELECTION_N_DOCS + LDA_EVAL_N_DOCS

if n_available < required_docs:
    raise ValueError(
        f"Only {n_available:,} training reviews available, "
        f"but {required_docs:,} are required."
    )

rng = np.random.RandomState(RANDOM_STATE)
indices = np.arange(n_available)
rng.shuffle(indices)

selection_idx = indices[:LDA_SELECTION_N_DOCS]
eval_idx = indices[
    LDA_SELECTION_N_DOCS:LDA_SELECTION_N_DOCS + LDA_EVAL_N_DOCS
]

selection_text_raw = train_text.iloc[selection_idx].tolist()
eval_text_raw = train_text.iloc[eval_idx].tolist()

print(f"LDA selection documents: {len(selection_text_raw):,}")
print(f"LDA evaluation documents: {len(eval_text_raw):,}")

In [ ]:
# CLEAN + SPACY PROCESS
selection_clean = [basic_clean(x) for x in selection_text_raw]
eval_clean = [basic_clean(x) for x in eval_text_raw]

print("Processing selection sample...")
selection_tokens_base = process_docs(selection_clean)

print("Processing evaluation sample...")
eval_tokens_base = process_docs(eval_clean)

del selection_clean, eval_clean
gc.collect()

# BIGRAM + TRIGRAM MODELS
bigram_model = Phrases(
    selection_tokens_base,
    min_count=PHRASE_MIN_COUNT,
    threshold=PHRASE_THRESHOLD
)
bigram = Phraser(bigram_model)

selection_tokens_bigram = [bigram[doc] for doc in selection_tokens_base]

trigram_model = Phrases(
    selection_tokens_bigram,
    min_count=PHRASE_MIN_COUNT,
    threshold=PHRASE_THRESHOLD
)
trigram = Phraser(trigram_model)

selection_tokens = [
    trigram[bigram[doc]]
    for doc in selection_tokens_base
]

eval_tokens = [
    trigram[bigram[doc]]
    for doc in eval_tokens_base
]

del selection_tokens_bigram, selection_tokens_base, eval_tokens_base
gc.collect()

#dictionary and corupus needed for gensim lda

dictionary = Dictionary(selection_tokens)

print("Initial vocabulary size:", len(dictionary))

dictionary.filter_extremes(
    no_below=VOCAB_NO_BELOW,
    no_above=VOCAB_NO_ABOVE,
    keep_n=LDA_MAX_FEATURES
)

print("Filtered vocabulary size:", len(dictionary))

if len(dictionary) == 0:
    raise ValueError("LDA vocabulary is empty.")

selection_corpus = [dictionary.doc2bow(doc) for doc in selection_tokens]
eval_corpus = [dictionary.doc2bow(doc) for doc in eval_tokens]

selection_tokens, selection_corpus = remove_empty_documents(
    selection_tokens,
    selection_corpus
)

eval_tokens, eval_corpus = remove_empty_documents(
    eval_tokens,
    eval_corpus
)

tf_selection = corpus_to_csr(selection_corpus, len(dictionary))
tf_eval = corpus_to_csr(eval_corpus, len(dictionary))

print(f"Selection documents: {len(selection_corpus):,}")
print(f"Evaluation documents: {len(eval_corpus):,}")
print(f"Vocabulary: {len(dictionary):,}")
print("Example document:", selection_tokens[0])


In [ ]:
# func to get coherence score
def topic_npmi_coherence(lda_model, tf_matrix, top_n=10, max_docs=20_000):
    if tf_matrix.shape[0] > max_docs:
        rng = np.random.RandomState(RANDOM_STATE)
        idx = rng.choice(tf_matrix.shape[0], size=max_docs, replace=False)
        X = tf_matrix[idx]
    else:
        X = tf_matrix

    X = (X > 0).astype(np.int8)
    n_docs = X.shape[0]

    word_doc_freq = np.asarray(X.sum(axis=0)).ravel()
    p_word = word_doc_freq / n_docs

    topic_word_indices = np.argsort(
        lda_model.components_,
        axis=1
    )[:, ::-1][:, :top_n]

    topic_scores = []

    for topic_indices in topic_word_indices:
        pair_scores = []

        for i in range(len(topic_indices)):
            for j in range(i + 1, len(topic_indices)):
                w1 = topic_indices[i]
                w2 = topic_indices[j]

                if p_word[w1] <= 0 or p_word[w2] <= 0:
                    continue

                cooc = X[:, w1].multiply(X[:, w2]).sum()

                if cooc == 0:
                    continue

                p12 = cooc / n_docs
                pmi = np.log(p12 / (p_word[w1] * p_word[w2]))
                npmi = pmi / (-np.log(p12))

                pair_scores.append(npmi)

        topic_scores.append(
            np.mean(pair_scores) if pair_scores else np.nan
        )

    return float(np.nanmean(topic_scores))

# func to get diversity score

def topic_diversity(lda_model, top_n=10):
    top_words = np.argsort(
        lda_model.components_,
        axis=1
    )[:, ::-1][:, :top_n]

    return len(np.unique(top_words)) / top_words.size

In [ ]:
from tqdm.auto import tqdm

# iterate overl all possible lda topic numbs as put in cofig, and gather results
results = []
candidate_models = {}

for n_topics in tqdm(N_TOPIC_CANDIDATES, desc="Testing topic counts"):
    print(f"Fitting LDA with {n_topics} topics...")

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        learning_method="online",
        batch_size=LDA_BATCH_SIZE,
        max_iter=LDA_SELECTION_MAX_ITER,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    lda.fit(tf_selection)

    perplexity = lda.perplexity(tf_eval)

    coherence = topic_npmi_coherence(
        lda,
        tf_selection,
        top_n=LDA_TOP_WORDS,
        max_docs=20_000
    )

    diversity = topic_diversity(
        lda,
        top_n=LDA_TOP_WORDS
    )

    results.append({
        "n_topics": n_topics,
        "perplexity": perplexity,
        "coherence": coherence,
        "topic_diversity": diversity
    })

    candidate_models[n_topics] = lda

    print(f"Perplexity:      {perplexity:,.2f}")
    print(f"Coherence:       {coherence:.4f}")
    print(f"Topic diversity: {diversity:.4f}")

In [ ]:
# final results
results_df = (
    pd.DataFrame(results)
      .sort_values("n_topics")
      .reset_index(drop=True)
)
print("\nLDA topic-count comparison:")
print(
    results_df.to_string(
        index=False,
        formatters={
            "perplexity": "{:,.2f}".format,
            "coherence": "{:.4f}".format,
            "topic_diversity": "{:.4f}".format
        }
    )
)

In [ ]:
# check outthe top words
feature_names = np.asarray([
    dictionary[i]
    for i in range(len(dictionary))
])

def print_topics(lda_model, feature_names, n_top_words=15):
    for topic_idx, topic in enumerate(lda_model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        words = feature_names[top_indices]
        print(f"Topic {topic_idx + 1:2d}: " + ", ".join(words))

for k in N_TOPIC_CANDIDATES:
    print("\n" + "=" * 80)
    print(f"{k} TOPICS")
    print("=" * 80)
    print_topics(
        candidate_models[k],
        feature_names,
        n_top_words=15
    )